#### Setup & Import Modules

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Import our custom modules
from src.config import (
    BASE_DIR, 
    ETHIOPIAN_REGIONS, 
    SPECIALTY_SCA_THRESHOLD
)
from src.data_loader import load_lots_data, validate_and_enrich_data
from src.utils import get_region_summary, filter_by_criteria

print("All modules imported successfully!")
print(f"Project Root: {BASE_DIR}")

Configuration loaded successfully!
Project Root: C:\Users\tohiba\Desktop\ethiopian-specialty-lot-intelligence
All modules imported successfully!
Project Root: C:\Users\tohiba\Desktop\ethiopian-specialty-lot-intelligence


#### Load and Validate Data

In [2]:
# Load data using our data_loader module
df = load_lots_data()
df = validate_and_enrich_data(df)

print(f"Dataset Loaded: {len(df)} coffee lots")
print(f"Time Period: Harvest {df['harvest_year'].unique()[0]}")
print(f"Specialty Lots (≥{SPECIALTY_SCA_THRESHOLD}): {df['is_specialty'].sum()}")

Loaded from pickle: 50 lots
Dataset Loaded: 50 coffee lots
Time Period: Harvest 2025
Specialty Lots (≥80.0): 50


#### Basic Overview

In [3]:
print("=== DATASET OVERVIEW ===")
print(f"Total Columns: {df.shape[1]}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n")

print("Sample Lots:")
display(df.head(3)[['lot_id', 'region', 'processing_method', 'sca_score', 
                   'grade_ecx', 'defects_per_300g', 'price_per_kg_usd']])

=== DATASET OVERVIEW ===
Total Columns: 25
Memory Usage: 0.04 MB

Sample Lots:


,lot_id,region,processing_method,sca_score,grade_ecx,defects_per_300g,price_per_kg_usd
0,ETH-2025-LIM-001,Kaffa,Washed,82.7,Grade 2,11,7.49
1,ETH-2025-KAF-002,Guji,Washed,89.8,Grade 2,9,7.39
2,ETH-2025-GUJ-003,Yirgacheffe,Natural,88.7,Grade 1,13,9.02


#### Quality & Compliance Analysis

In [4]:
print("=== QUALITY & COMPLIANCE ANALYSIS ===\n")

# Summary statistics
quality_summary = df.groupby('quality_tier').agg({
    'sca_score': ['count', 'mean', 'min', 'max'],
    'defects_per_300g': 'mean',
    'moisture_pct': 'mean',
    'available_quantity_kg': 'sum'
}).round(2)

print("By Quality Tier:")
display(quality_summary)

# ECX Compliance
ecx_compliance = df['ecx_compliant'].value_counts()
print(f"\nECX Compliant Lots: {ecx_compliance.get(True, 0)} / {len(df)}")

=== QUALITY & COMPLIANCE ANALYSIS ===



C:\Users\tohiba\AppData\Local\Temp\ipykernel_15844\4088651694.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  quality_summary = df.groupby('quality_tier').agg({


By Quality Tier:


sca_score                    defects_per_300g moisture_pct  \
                 count   mean   min   max             mean         mean   
quality_tier                                                              
Good                 0    NaN   NaN   NaN              NaN          NaN   
Very Good           14  82.93  82.1  84.7             4.50        10.83   
Excellent           16  86.72  85.5  88.0             6.62        10.86   
Outstanding         20  90.23  88.2  91.4             8.80        10.99   

             available_quantity_kg  
                               sum  
quality_tier                        
Good                           0.0  
Very Good                  97800.0  
Excellent                 103380.0  
Outstanding               145260.0


ECX Compliant Lots: 22 / 50


#### Regional Performance

In [5]:
print("=== REGIONAL PERFORMANCE ===")

region_summary = get_region_summary(df)
display(region_summary)

# Top regions by average score
top_regions = region_summary.sort_values(('sca_score', 'mean'), ascending=False)
print("\nTop Regions by Average SCA Score:")
print(top_regions)

=== REGIONAL PERFORMANCE ===


sca_score                   available_quantity_kg  \
                 mean   max   min count                   sum   
region                                                          
Guji            87.75  91.2  82.8    10               72960.0   
Kaffa           86.80  91.2  82.7    10               61860.0   
Limmu           85.99  91.4  82.1    13               84300.0   
Sidama          87.35  89.1  86.0     4               26700.0   
Yirgacheffe     87.72  91.4  82.4    13              100620.0   

            price_per_kg_usd is_specialty  
                        mean          sum  
region                                     
Guji                    7.45           10  
Kaffa                   7.70           10  
Limmu                   7.92           13  
Sidama                  8.44            4  
Yirgacheffe             7.89           13


Top Regions by Average SCA Score:
            sca_score                   available_quantity_kg  \
                 mean   max   min count                   sum   
region                                                          
Guji            87.75  91.2  82.8    10               72960.0   
Yirgacheffe     87.72  91.4  82.4    13              100620.0   
Sidama          87.35  89.1  86.0     4               26700.0   
Kaffa           86.80  91.2  82.7    10               61860.0   
Limmu           85.99  91.4  82.1    13               84300.0   

            price_per_kg_usd is_specialty  
                        mean          sum  
region                                     
Guji                    7.45           10  
Yirgacheffe             7.89           13  
Sidama                  8.44            4  
Kaffa                   7.70           10  
Limmu                   7.92           13  


#### Processing Method Analysis

In [6]:
print("=== PROCESSING METHOD ANALYSIS ===\n")

processing_analysis = df.groupby('processing_method').agg({
    'sca_score': ['mean', 'max', 'count'],
    'defects_per_300g': 'mean',
    'price_per_kg_usd': 'mean',
    'available_quantity_kg': 'sum'
}).round(2)

display(processing_analysis)

=== PROCESSING METHOD ANALYSIS ===



sca_score             defects_per_300g price_per_kg_usd  \
                       mean   max count             mean             mean   
processing_method                                                           
Honey                 86.37  91.4     3             6.67             6.12   
Natural               87.68  91.4    24             7.54             8.14   
Washed                86.50  91.4    23             6.26             7.71   

                  available_quantity_kg  
                                    sum  
processing_method                        
Honey                           23160.0  
Natural                        159120.0  
Washed                         164160.0

#### Price vs Quality Relationship

In [7]:
print("=== PRICE vs QUALITY ANALYSIS ===\n")

# Correlation
correlation = df['sca_score'].corr(df['price_per_kg_usd'])
print(f"Correlation between SCA Score and Price: {correlation:.3f}\n")

# Price tiers
df['price_tier'] = pd.qcut(df['price_per_kg_usd'], q=4, labels=['Budget', 'Mid', 'Premium', 'Ultra Premium'])

price_quality = df.groupby('price_tier').agg({
    'sca_score': ['mean', 'min', 'max'],
    'quantity_bags': 'sum'
}).round(2)

display(price_quality)

=== PRICE vs QUALITY ANALYSIS ===

Correlation between SCA Score and Price: 0.122



C:\Users\tohiba\AppData\Local\Temp\ipykernel_15844\2856870141.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  price_quality = df.groupby('price_tier').agg({


sca_score             quantity_bags
                   mean   min   max           sum
price_tier                                       
Budget            87.04  82.1  91.4          1353
Mid               86.89  82.5  91.4          1458
Premium           87.13  82.1  90.6          1582
Ultra Premium     87.18  82.8  91.4          1381

#### Traceability & Sustainability Insights

In [8]:
print("=== TRACEABILITY & SUSTAINABILITY ===\n")

trace_summary = df.groupby('traceability_level').agg({
    'sca_score': 'mean',
    'price_per_kg_usd': 'mean',
    'available_quantity_kg': 'sum'
}).round(2)

print("By Traceability Level:")
display(trace_summary)

sust_summary = df.groupby('sustainability_cert').agg({
    'sca_score': 'mean',
    'price_per_kg_usd': 'mean'
}).round(2)

print("\nBy Sustainability Certification:")
display(sust_summary)

=== TRACEABILITY & SUSTAINABILITY ===

By Traceability Level:


,sca_score,price_per_kg_usd,available_quantity_kg
traceability_level,,,
Exporter,86.71,7.23,63720.0
Full,87.18,7.88,164400.0
Washing Station,87.05,8.00,118320.0



By Sustainability Certification:


,sca_score,price_per_kg_usd
sustainability_cert,,
None,86.73,7.33
Organic,85.82,7.94
Rainforest Alliance,88.55,8.43
Volcafe Verified,87.08,7.57


#### Final Summary & Insights

In [9]:
print("=== KEY INSIGHTS FOR VOLCAFE SPECIALTY SOURCING ===\n")

print(f"1. Total Specialty Lots Available : {df['is_specialty'].sum()}")
print(f"2. Highest Scoring Lot           : {df['sca_score'].max()} pts")
print(f"3. Average Price (Specialty)     : ${df[df['is_specialty']]['price_per_kg_usd'].mean():.2f}/kg")
print(f"4. Best Region (Avg Score)       : {region_summary[('sca_score','mean')].idxmax()}")
print(f"5. Most Common Processing        : {df['processing_method'].mode()[0]}")

# Recommendation simulation
top_lots = df.nlargest(5, 'sca_score')[['lot_id', 'region', 'supplier_name', 'sca_score', 'price_per_kg_usd']]
print("\nTop 5 Recommended Lots for International Clients:")
display(top_lots)

=== KEY INSIGHTS FOR VOLCAFE SPECIALTY SOURCING ===

1. Total Specialty Lots Available : 50
2. Highest Scoring Lot           : 91.4 pts
3. Average Price (Specialty)     : $7.82/kg
4. Best Region (Avg Score)       : Guji
5. Most Common Processing        : Natural

Top 5 Recommended Lots for International Clients:


,lot_id,region,supplier_name,sca_score,price_per_kg_usd
5,ETH-2025-SID-006,Limmu,Guji Highland,91.4,9.65
10,ETH-2025-LIM-011,Limmu,Daye Bensa,91.4,7.13
39,ETH-2025-SID-040,Yirgacheffe,Daye Bensa,91.4,7.67
33,ETH-2025-YIR-034,Yirgacheffe,Red Cherry Export,91.2,5.41
35,ETH-2025-KAF-036,Kaffa,Red Cherry Export,91.2,6.80
